In [ ]:
import numpy as np
import optuna
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

from students.marchenko.lesson3 import Exercise


def normalize_data(X_train, X_val, X_test):
    feature_means = X_train.mean(axis=0)
    feature_stds = X_train.std(axis=0)
    feature_stds[feature_stds == 0] = 1.0

    X_train_norm = (X_train - feature_means) / feature_stds
    X_val_norm = (X_val - feature_means) / feature_stds
    X_test_norm = (X_test - feature_means) / feature_stds

    return X_train_norm, X_val_norm, X_test_norm


digits = load_digits()
X = digits.data.astype(np.float32)
y = digits.target.astype(np.int64)

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.4, random_state=100, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=100, stratify=y_temp)

X_train_norm, X_val_norm, X_test_norm = normalize_data(X_train, X_val, X_test)

Model = Exercise.create_model
LinearLayer = Exercise.create_linear_layer
ReLULayer = Exercise.create_relu_layer
CrossEntropyLoss = Exercise.create_cross_entropy_loss


def objective(trial):
    hidden_size = trial.suggest_int("hidden_size", 16, 128)
    lr = trial.suggest_float("lr", 1e-4, 0.1, log=True)
    batch_size = trial.suggest_int("batch_size", 8, 128)
    n_epoch = trial.suggest_int("n_epoch", 10, 50)

    rng = np.random.default_rng(trial.number)
    model = Model(
        LinearLayer(64, hidden_size, rng=rng),
        ReLULayer(),
        LinearLayer(hidden_size, 10, rng=rng),
    )
    loss = CrossEntropyLoss()

    Exercise.train_model(model, loss, X_train_norm, y_train, lr, n_epoch, batch_size)

    val_pred = model.forward(X_val_norm)
    val_acc = np.mean(np.argmax(val_pred, axis=1) == y_val)

    return val_acc


study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=50)

print("\nЛучшие параметры:", study.best_params)
print(f"Лучшая валидационная точность: {study.best_value:.4f}")

# Финальное обучение с лучшими параметрами
best = study.best_params
final_rng = np.random.default_rng(42)
final_model = Model(
    LinearLayer(64, best["hidden_size"], rng=final_rng),
    ReLULayer(),
    LinearLayer(best["hidden_size"], 10, rng=final_rng),
)

Exercise.train_model(
    model=final_model,
    loss=CrossEntropyLoss(),
    x=X_train_norm,
    y=y_train,
    lr=best["lr"],
    n_epoch=best["n_epoch"],
    batch_size=best["batch_size"],
)

test_predictions = final_model.forward(X_test_norm)
test_predicted_classes = np.argmax(test_predictions, axis=1)
test_acc = np.mean(test_predicted_classes == y_test)

print(f"Точность на тесте: {test_acc:.4f}")